In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.secret.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [4]:
# ============================================================
# Lê o streaming da camada Bronze
# ============================================================
df_raw = (
    spark.readStream
    .schema("regiao STRING, codigo_linha INT, timestamp STRING, raw_json STRING")
    .json("s3a://raw/sptrans/previsao/")
)

In [5]:
# ============================================================
# Explode o JSON interno (raw_json)
# ============================================================
from pyspark.sql.functions import from_json, explode, col
from pyspark.sql.types import *

# Define o schema do conteúdo JSON retornado pela API
schema_previsao = StructType([
    StructField("hr", StringType()),
    StructField("ps", ArrayType(
        StructType([
            StructField("cp", StringType()),
            StructField("np", StringType()),
            StructField("py", DoubleType()),
            StructField("px", DoubleType()),
            StructField("vs", ArrayType(
                StructType([
                    StructField("p", StringType()),
                    StructField("t", StringType()),
                    StructField("a", BooleanType())
                ])
            ))
        ])
    ))
])

df_parsed = (
    df_raw
    .withColumn("json_data", from_json(col("raw_json"), schema_previsao))
    .withColumn("parada", explode(col("json_data.ps")))
    .withColumn("veiculo", explode(col("parada.vs")))
)


In [6]:
# ============================================================
# Seleciona e renomeia colunas relevantes
# ============================================================
df_silver = (
    df_parsed.select(
        "regiao",
        "codigo_linha",
        "timestamp",
        col("json_data.hr").alias("hora_coleta"),
        col("parada.cp").alias("codigo_parada"),
        col("parada.np").alias("nome_parada"),
        col("parada.py").alias("latitude"),
        col("parada.px").alias("longitude"),
        col("veiculo.p").alias("prefixo_veiculo"),
        col("veiculo.t").alias("hora_prevista"),
        col("veiculo.a").alias("acessivel")
    )
)

In [10]:
# ============================================================
# Escreve continuamente na camada Silver (Parquet)
# ============================================================
(
    df_silver.writeStream
    .format("parquet")
    .option("checkpointLocation", "s3a://silver/sptrans/checkpoints/previsao_chegada/")
    .option("path", "s3a://silver/previsao_chegada/")
    .outputMode("append")
    .trigger(processingTime="2 minutes")  # Atualiza a cada 2 minutos
    .start()
)